# ADAPT-IDS: Full Pipeline Verification Notebook

**Adaptive Intrusion Detection Under Concept and Feature Drift**

This notebook provides **concrete, runnable proof** that every component of the ADAPT-IDS pipeline works correctly. It uses synthetic CIC-IDS2017-like data so it runs anywhere — no dataset download required.

### What this verifies:
1. **Environment & Dependencies** — all packages install and import
2. **Data Generation** — realistic synthetic network flow data
3. **Preprocessing Pipeline** — cleaning, NaN/inf handling, label mapping, leakage prevention
4. **Model Training** — LightGBM, Random Forest, and LSTM classifiers
5. **Evaluation** — IDS metrics (F1, precision, recall, FPR, FNR, MCC)
6. **Temporal vs Random Splitting** — demonstrates drift degradation
7. **Drift Detection** — ADWIN, DDM, EDDM, Page-Hinkley detectors
8. **Synthetic Drift Injection** — sudden, gradual, incremental, recurring
9. **Streaming Simulation** — CSVStream and BatchStream
10. **Adaptation Strategies** — static, periodic, drift-triggered retraining
11. **Active Learning** — uncertainty, margin, and random sampling
12. **End-to-End Adaptive Pipeline** — full streaming drift detection + retraining
13. **Visualizations** — publication-quality plots generated inline

### Platform Support
- **Google Colab** (Linux, T4/A100 GPU)
- **macOS Apple Silicon** (M1/M2/M3/M4)
- **Windows** (x86_64)

---

## 0. Environment Setup

Detects the platform and installs all dependencies. On Colab, clones the repository.

In [ ]:
import sys
import platform
import os

print(f"Python:   {sys.version}")
print(f"Platform: {platform.system()} {platform.machine()}")
print(f"OS:       {platform.platform()}")

IN_COLAB = 'google.colab' in sys.modules
IS_MAC_ARM = platform.system() == 'Darwin' and platform.machine() == 'arm64'
IS_WINDOWS = platform.system() == 'Windows'

print(f"\nRunning on: {'Google Colab' if IN_COLAB else 'macOS ARM' if IS_MAC_ARM else 'Windows' if IS_WINDOWS else 'Linux/Other'}")

In [ ]:
if IN_COLAB:
    import importlib
    _needs_restart = False

    # Test if numpy/sklearn are healthy before touching anything
    try:
        import numpy as _np
        import sklearn as _sk
        _np.testing._private.utils  # triggers the _blas_supports_fpe path
        print("numpy/sklearn already healthy — skipping reinstall")
    except (AttributeError, ImportError, Exception) as e:
        print(f"Detected broken numpy/sklearn ({e.__class__.__name__}), fixing...")
        _needs_restart = True

    !git clone https://github.com/aemmanuval/ADAPT-IDS.git /content/ADAPT-IDS 2>/dev/null || true
    os.chdir('/content/ADAPT-IDS')

    if _needs_restart:
        # Fully remove the broken numpy and reinstall a clean stack
        !pip uninstall -y numpy 2>/dev/null
        !pip install "numpy>=1.26,<2.1" 2>&1 | tail -1
        !pip install "scipy>=1.12,<2.0" "scikit-learn>=1.4,<2.0" "pandas>=2.1,<3.0" 2>&1 | tail -1

    !pip install -q lightgbm "river>=0.21,<1.0" pyyaml matplotlib seaborn joblib tqdm pyarrow 2>&1 | tail -1

    # PyTorch: keep Colab's pre-installed version if available
    try:
        import torch
    except ImportError:
        !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

    sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

    if _needs_restart:
        print("\n*** Restarting runtime to load clean numpy. After restart, click Runtime > Run all again. ***")
        import time; time.sleep(2)
        from google.colab import runtime
        runtime.unassign()  # restarts the Colab runtime cleanly
    else:
        print(f"Working directory: {os.getcwd()}")
        print(f"Source path: {os.path.join(os.getcwd(), 'src')}")
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if not os.path.exists(os.path.join(repo_root, 'src', 'adaptive_ids')):
        repo_root = os.getcwd()
    os.chdir(repo_root)
    !pip install -q numpy pandas scikit-learn lightgbm river pyyaml matplotlib seaborn joblib tqdm scipy pyarrow torch 2>/dev/null || pip install numpy pandas scikit-learn lightgbm river pyyaml matplotlib seaborn joblib tqdm scipy pyarrow torch

    sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
    print(f"Working directory: {os.getcwd()}")
    print(f"Source path: {os.path.join(os.getcwd(), 'src')}")

In [ ]:
# Quick sanity check: verify critical imports work after install
import numpy as _np_check
import pandas as _pd_check
import sklearn as _sk_check
print(f"numpy {_np_check.__version__}, pandas {_pd_check.__version__}, sklearn {_sk_check.__version__} — OK")

## 1. Dependency & Import Verification

Imports every module used in the project and prints version info.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
import river
import yaml
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import scipy
import joblib

deps = {
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'lightgbm': lgb.__version__,
    'river': river.__version__,
    'matplotlib': matplotlib.__version__,
    'seaborn': sns.__version__,
    'torch': torch.__version__,
    'scipy': scipy.__version__,
}

print("=" * 50)
print("  DEPENDENCY VERSIONS")
print("=" * 50)
for pkg, ver in deps.items():
    print(f"  {pkg:20s} {ver}")
print("=" * 50)

if IS_MAC_ARM:
    print(f"\nApple Silicon: MPS available = {torch.backends.mps.is_available()}")
elif torch.cuda.is_available():
    print(f"\nCUDA: {torch.cuda.get_device_name(0)}")
else:
    print("\nRunning on CPU")

In [ ]:
from adaptive_ids.config.settings import load_config, get_project_root
from adaptive_ids.preprocessing.pipeline import PreprocessingPipeline
from adaptive_ids.models.baseline import BaselineIDS
from adaptive_ids.models.lstm_ids import LSTMClassifier
from adaptive_ids.evaluation.metrics import compute_metrics, compute_windowed_metrics
from adaptive_ids.evaluation.temporal import temporal_split, random_split
from adaptive_ids.streaming.stream import CSVStream, BatchStream
from adaptive_ids.drift.detectors import ADWINDetector, DDMDetector, EDDMDetector, PageHinkleyDetector, UnsupervisedDriftDetector, create_detector
from adaptive_ids.drift.synthetic import SyntheticDriftGenerator
from adaptive_ids.adaptation.strategies import StaticStrategy, PeriodicStrategy, DriftTriggeredStrategy, AdaptiveModelManager
from adaptive_ids.adaptation.active_learning import UncertaintySampling, MarginSampling, RandomSampling, ActiveLearningManager
from adaptive_ids.features.selection import select_available_features, compute_feature_stats
from adaptive_ids.utils.reproducibility import set_global_seed

print("All ADAPT-IDS modules imported successfully!")
print(f"Project root: {get_project_root()}")

## 2. Synthetic Data Generation

Creates a realistic CIC-IDS2017-like dataset with:
- 5,000 network flows across 5 days (temporal ordering matters!)
- Realistic feature distributions (packet counts, byte sizes, timing, flags)
- Class imbalance: ~70% benign, 30% mixed attack types
- Deliberate NaN/Inf values to test preprocessing
- Temporal drift: attack patterns shift across days

In [ ]:
set_global_seed(42)
rng = np.random.RandomState(42)

N_SAMPLES = 5000

timestamps = pd.date_range("2017-07-03 09:00", periods=N_SAMPLES, freq="30s")

benign_mask = np.zeros(N_SAMPLES, dtype=bool)
benign_mask[:3500] = True
rng.shuffle(benign_mask)

day_index = (np.arange(N_SAMPLES) // 1000)

labels = []
attack_types = ['DDoS', 'PortScan', 'Bot', 'FTP-Patator', 'SSH-Patator']
for i in range(N_SAMPLES):
    if benign_mask[i]:
        labels.append('BENIGN')
    else:
        day = day_index[i]
        weights = np.roll([0.4, 0.3, 0.15, 0.1, 0.05], day)
        labels.append(rng.choice(attack_types, p=weights))

def gen_feature(benign_params, attack_params):
    vals = np.zeros(N_SAMPLES)
    vals[benign_mask] = rng.exponential(benign_params[0], benign_mask.sum()) + benign_params[1]
    vals[~benign_mask] = rng.exponential(attack_params[0], (~benign_mask).sum()) + attack_params[1]
    return np.abs(vals)

df = pd.DataFrame({
    'Timestamp': timestamps,
    'Flow Duration': gen_feature((50000, 1000), (5000, 100)),
    'Total Fwd Packets': gen_feature((5, 1), (50, 10)).astype(float),
    'Total Backward Packets': gen_feature((3, 1), (20, 5)).astype(float),
    'Total Length of Fwd Packets': gen_feature((500, 50), (5000, 500)),
    'Total Length of Bwd Packets': gen_feature((300, 30), (3000, 300)),
    'Flow Bytes/s': gen_feature((10000, 500), (100000, 5000)),
    'Flow Packets/s': gen_feature((100, 5), (1000, 50)),
    'Flow IAT Mean': gen_feature((5000, 100), (500, 10)),
    'Flow IAT Std': gen_feature((3000, 100), (300, 10)),
    'Fwd IAT Mean': gen_feature((4000, 100), (400, 10)),
    'Bwd IAT Mean': gen_feature((4000, 100), (400, 10)),
    'Fwd Packet Length Mean': gen_feature((200, 20), (100, 10)),
    'Bwd Packet Length Mean': gen_feature((150, 15), (80, 8)),
    'SYN Flag Count': rng.binomial(1, np.where(benign_mask, 0.3, 0.8), N_SAMPLES).astype(float),
    'ACK Flag Count': rng.binomial(1, np.where(benign_mask, 0.7, 0.3), N_SAMPLES).astype(float),
    'Init_Win_bytes_forward': rng.randint(0, 65535, N_SAMPLES).astype(float),
    'Init_Win_bytes_backward': rng.randint(0, 65535, N_SAMPLES).astype(float),
    'Label': labels,
})

drift_start = 3500
for col in ['Flow Bytes/s', 'Flow Packets/s', 'Total Fwd Packets']:
    df.loc[drift_start:, col] *= rng.uniform(1.5, 3.0)

inf_idx = rng.choice(N_SAMPLES, 15, replace=False)
df.loc[inf_idx[:5], 'Flow Bytes/s'] = np.inf
df.loc[inf_idx[5:10], 'Flow Packets/s'] = -np.inf
nan_idx = rng.choice(N_SAMPLES, 20, replace=False)
df.loc[nan_idx[:10], 'Flow Duration'] = np.nan
df.loc[nan_idx[10:], 'Fwd IAT Mean'] = np.nan

print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['Label'].value_counts())
print(f"\nInf values:  {np.isinf(df.select_dtypes('number')).sum().sum()}")
print(f"NaN values:  {df.isna().sum().sum()}")
print(f"\nDate range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")
df.head()

## 3. Preprocessing Pipeline Verification

Tests the full preprocessing pipeline: infinity handling, NaN imputation, label mapping, constant column removal, and leakage prevention.

In [ ]:
config = load_config()
pipeline = PreprocessingPipeline(config)

X_df, X, y = pipeline.fit_transform(df)

print("=" * 50)
print("  PREPROCESSING RESULTS")
print("=" * 50)
print(f"  Input shape:    {df.shape}")
print(f"  Output X shape: {X.shape}")
print(f"  Output y shape: {y.shape}")
print(f"  Features:       {X.shape[1]}")
print(f"  Feature names:  {pipeline.feature_columns[:5]}... ({len(pipeline.feature_columns)} total)")
print(f"  Labels:         {np.unique(y)}")
print(f"  NaN in X:       {np.isnan(X).sum()}")
print(f"  Inf in X:       {np.isinf(X).sum()}")
print(f"  Pipeline fitted: {pipeline._is_fitted}")

assert np.isnan(X).sum() == 0, "FAIL: NaN values remain after preprocessing!"
assert np.isinf(X).sum() == 0, "FAIL: Inf values remain after preprocessing!"
assert set(y) == {'ATTACK', 'BENIGN'}, f"FAIL: Expected binary labels, got {set(y)}"
assert X.shape[0] == len(y), "FAIL: X and y row count mismatch!"
assert pipeline._is_fitted, "FAIL: Pipeline not marked as fitted!"

print("\n  ALL PREPROCESSING CHECKS PASSED")
print("=" * 50)

In [ ]:
test_subset = df.sample(500, random_state=99)
X_df_t, X_t, y_t = pipeline.transform(test_subset)

print(f"Transform on unseen data: X={X_t.shape}, y={y_t.shape}")
assert X_t.shape[1] == X.shape[1], f"FAIL: Feature count mismatch! Train={X.shape[1]}, Test={X_t.shape[1]}"
assert np.isnan(X_t).sum() == 0, "FAIL: NaN in transformed data!"
print("Leakage-safe transform: PASSED")

## 4. Temporal vs Random Splitting

Demonstrates the critical difference between IID (random) and temporal (chronological) evaluation. This is the core motivation for the research — temporal splits reveal drift degradation.

In [ ]:
temporal_splits = temporal_split(df, train_fraction=0.7, validation_fraction=0.1, test_fraction=0.2)
random_splits = random_split(df, train_fraction=0.8, test_fraction=0.2, random_seed=42, stratify_column='Label')

print("TEMPORAL SPLIT:")
for name, split_df in temporal_splits.items():
    label_dist = split_df['Label'].value_counts().to_dict()
    ts_range = f"{split_df['Timestamp'].min()} → {split_df['Timestamp'].max()}"
    print(f"  {name:12s}: {len(split_df):5d} rows | {label_dist} | {ts_range}")

print(f"\nRANDOM SPLIT:")
for name, split_df in random_splits.items():
    label_dist = split_df['Label'].value_counts().to_dict()
    print(f"  {name:12s}: {len(split_df):5d} rows | {label_dist}")

train_max_ts = pd.to_datetime(temporal_splits['train']['Timestamp']).max()
test_min_ts = pd.to_datetime(temporal_splits['test']['Timestamp']).min()
assert train_max_ts < test_min_ts, "FAIL: Temporal leakage detected!"
print(f"\nTemporal integrity: train_max={train_max_ts} < test_min={test_min_ts} -> PASSED")

## 5. Model Training & Evaluation

### 5a. LightGBM (Primary Model)

In [ ]:
pipeline_rnd = PreprocessingPipeline(config)
_, X_train_rnd, y_train_rnd = pipeline_rnd.fit_transform(random_splits['train'])
_, X_test_rnd, y_test_rnd = pipeline_rnd.transform(random_splits['test'])

lgbm_model = BaselineIDS(
    algorithm='lightgbm',
    params={'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'num_leaves': 63,
            'class_weight': 'balanced', 'verbose': -1, 'n_jobs': 1, 'num_threads': 1},
    random_seed=42
)
lgbm_model.fit(X_train_rnd, y_train_rnd)

y_pred_rnd = lgbm_model.predict(X_test_rnd)
rnd_metrics = compute_metrics(y_test_rnd, y_pred_rnd, positive_label='ATTACK')

print("=" * 60)
print("  LightGBM — RANDOM SPLIT (IID baseline)")
print("=" * 60)
print(f"  Accuracy:   {rnd_metrics['accuracy']:.4f}")
print(f"  Precision:  {rnd_metrics['precision']:.4f}")
print(f"  Recall:     {rnd_metrics['recall']:.4f}")
print(f"  F1 Score:   {rnd_metrics['f1']:.4f}")
print(f"  MCC:        {rnd_metrics['mcc']:.4f}")
print(f"  FPR:        {rnd_metrics['fpr']:.4f}")
print(f"  FNR:        {rnd_metrics['fnr']:.4f}")
print(f"  Training:   {lgbm_model.training_time:.2f}s")
print(f"  Confusion:  {rnd_metrics['confusion_matrix']}")

assert rnd_metrics['f1'] > 0.7, f"FAIL: LightGBM F1 too low on random split: {rnd_metrics['f1']:.4f}"
print(f"\n  F1 > 0.70: PASSED ({rnd_metrics['f1']:.4f})")
print("=" * 60)

In [ ]:
pipeline_tmp = PreprocessingPipeline(config)
_, X_train_tmp, y_train_tmp = pipeline_tmp.fit_transform(temporal_splits['train'])
_, X_test_tmp, y_test_tmp = pipeline_tmp.transform(temporal_splits['test'])

lgbm_temporal = BaselineIDS(
    algorithm='lightgbm',
    params={'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'num_leaves': 63,
            'class_weight': 'balanced', 'verbose': -1, 'n_jobs': 1, 'num_threads': 1},
    random_seed=42
)
lgbm_temporal.fit(X_train_tmp, y_train_tmp)

y_pred_tmp = lgbm_temporal.predict(X_test_tmp)
tmp_metrics = compute_metrics(y_test_tmp, y_pred_tmp, positive_label='ATTACK')

print("=" * 60)
print("  LightGBM — TEMPORAL SPLIT (drift-exposed)")
print("=" * 60)
print(f"  Accuracy:   {tmp_metrics['accuracy']:.4f}")
print(f"  Precision:  {tmp_metrics['precision']:.4f}")
print(f"  Recall:     {tmp_metrics['recall']:.4f}")
print(f"  F1 Score:   {tmp_metrics['f1']:.4f}")
print(f"  MCC:        {tmp_metrics['mcc']:.4f}")
print(f"  FPR:        {tmp_metrics['fpr']:.4f}")
print(f"  FNR:        {tmp_metrics['fnr']:.4f}")

delta_f1 = rnd_metrics['f1'] - tmp_metrics['f1']
print(f"\n  F1 drop from random to temporal: {delta_f1:+.4f}")
if delta_f1 > 0.01:
    print("  -> Drift degradation detected (expected behavior!)")
else:
    print("  -> Minimal drift impact on synthetic data (OK for verification)")
print("=" * 60)

### 5b. Random Forest

In [ ]:
rf_model = BaselineIDS(
    algorithm='random_forest',
    params={'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 5,
            'min_samples_leaf': 2, 'class_weight': 'balanced', 'n_jobs': 1},
    random_seed=42
)
rf_model.fit(X_train_rnd, y_train_rnd)

y_pred_rf = rf_model.predict(X_test_rnd)
rf_metrics = compute_metrics(y_test_rnd, y_pred_rf, positive_label='ATTACK')

print("=" * 60)
print("  Random Forest — RANDOM SPLIT")
print("=" * 60)
print(f"  Accuracy:   {rf_metrics['accuracy']:.4f}")
print(f"  Precision:  {rf_metrics['precision']:.4f}")
print(f"  Recall:     {rf_metrics['recall']:.4f}")
print(f"  F1 Score:   {rf_metrics['f1']:.4f}")
print(f"  MCC:        {rf_metrics['mcc']:.4f}")
print(f"  Training:   {rf_model.training_time:.2f}s")

assert rf_metrics['f1'] > 0.5, f"FAIL: RF F1 too low: {rf_metrics['f1']:.4f}"
print(f"\n  F1 > 0.50: PASSED ({rf_metrics['f1']:.4f})")
print("=" * 60)

### 5c. LSTM Deep Learning Classifier

Bidirectional LSTM with attention mechanism. Uses MPS on Apple Silicon, CUDA on GPU-equipped machines, CPU otherwise.

In [ ]:
from sklearn.preprocessing import StandardScaler

lstm = LSTMClassifier(
    hidden_size=64,
    num_layers=2,
    dropout=0.3,
    learning_rate=0.001,
    epochs=10,
    batch_size=256,
    seq_len=8,
    random_seed=42
)

if IS_MAC_ARM and torch.backends.mps.is_available():
    print("Using Apple MPS acceleration")
elif torch.cuda.is_available():
    lstm.device = torch.device('cuda')
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")

lstm.fit(X_train_rnd, y_train_rnd)

y_pred_lstm = lstm.predict(X_test_rnd)
lstm_metrics = compute_metrics(y_test_rnd, y_pred_lstm, positive_label='ATTACK')

print(f"\n{'=' * 60}")
print("  LSTM — RANDOM SPLIT")
print("=" * 60)
print(f"  Accuracy:   {lstm_metrics['accuracy']:.4f}")
print(f"  Precision:  {lstm_metrics['precision']:.4f}")
print(f"  Recall:     {lstm_metrics['recall']:.4f}")
print(f"  F1 Score:   {lstm_metrics['f1']:.4f}")
print(f"  MCC:        {lstm_metrics['mcc']:.4f}")
print(f"  Training:   {lstm.training_time:.2f}s")
print(f"  Metadata:   {lstm.metadata}")
print("=" * 60)

### 5d. Model Comparison Summary

In [ ]:
comparison = pd.DataFrame({
    'Model': ['LightGBM (Random)', 'LightGBM (Temporal)', 'Random Forest (Random)', 'LSTM (Random)'],
    'F1': [rnd_metrics['f1'], tmp_metrics['f1'], rf_metrics['f1'], lstm_metrics['f1']],
    'Precision': [rnd_metrics['precision'], tmp_metrics['precision'], rf_metrics['precision'], lstm_metrics['precision']],
    'Recall': [rnd_metrics['recall'], tmp_metrics['recall'], rf_metrics['recall'], lstm_metrics['recall']],
    'MCC': [rnd_metrics['mcc'], tmp_metrics['mcc'], rf_metrics['mcc'], lstm_metrics['mcc']],
    'FPR': [rnd_metrics.get('fpr', '-'), tmp_metrics.get('fpr', '-'), rf_metrics.get('fpr', '-'), lstm_metrics.get('fpr', '-')],
    'FNR': [rnd_metrics.get('fnr', '-'), tmp_metrics.get('fnr', '-'), rf_metrics.get('fnr', '-'), lstm_metrics.get('fnr', '-')],
})

print("\n" + "=" * 90)
print("  MODEL COMPARISON")
print("=" * 90)
print(comparison.to_string(index=False, float_format='{:.4f}'.format))
print("=" * 90)

### 5e. Model Save / Load Verification

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    lgbm_path = os.path.join(tmpdir, 'lgbm_model.joblib')
    lgbm_model.save(lgbm_path)
    loaded = BaselineIDS.load(lgbm_path)
    y_pred_loaded = loaded.predict(X_test_rnd)
    match = np.mean(y_pred_rnd == y_pred_loaded)
    print(f"LightGBM save/load — prediction match: {match:.4f}")
    assert match == 1.0, f"FAIL: Loaded model predictions differ! Match={match}"
    
    rf_path = os.path.join(tmpdir, 'rf_model.joblib')
    rf_model.save(rf_path)
    loaded_rf = BaselineIDS.load(rf_path)
    y_pred_loaded_rf = loaded_rf.predict(X_test_rnd)
    match_rf = np.mean(y_pred_rf == y_pred_loaded_rf)
    print(f"Random Forest save/load — prediction match: {match_rf:.4f}")
    assert match_rf == 1.0, f"FAIL: Loaded RF model predictions differ! Match={match_rf}"
    
    lstm_path = os.path.join(tmpdir, 'lstm_model.joblib')
    lstm.save(lstm_path)
    loaded_lstm = LSTMClassifier.load(lstm_path)
    y_pred_loaded_lstm = loaded_lstm.predict(X_test_rnd)
    match_lstm = np.mean(y_pred_lstm == y_pred_loaded_lstm)
    print(f"LSTM save/load — prediction match: {match_lstm:.4f}")
    assert match_lstm == 1.0, f"FAIL: Loaded LSTM predictions differ! Match={match_lstm}"

print("\nAll model save/load checks: PASSED")

## 6. Windowed Metrics (Performance Over Time)

In [ ]:
windowed = compute_windowed_metrics(y_test_tmp, y_pred_tmp, window_size=200, positive_label='ATTACK')

print(f"Windows computed: {len(windowed)}")
print(f"\nSample window metrics:")
for w in windowed[:3]:
    print(f"  Window {w['window_id']}: F1={w['f1']:.4f}, Precision={w['precision']:.4f}, Recall={w['recall']:.4f}")

f1_values = [w['f1'] for w in windowed]
print(f"\nF1 range: [{min(f1_values):.4f}, {max(f1_values):.4f}]")
print(f"F1 mean:  {np.mean(f1_values):.4f}")
print(f"F1 std:   {np.std(f1_values):.4f}")

## 7. Drift Detection Verification

Tests all four drift detectors with a controlled distribution shift.

In [ ]:
def test_detector(detector, name, stable_p=0.05, drift_p=0.6, n_stable=500, n_drift=500):
    """Feed a detector a stable stream followed by a high-error stream."""
    rng = np.random.RandomState(42)
    drift_pos = None
    
    for i in range(n_stable):
        detector.update(float(rng.binomial(1, stable_p)))
    
    for i in range(n_drift):
        detector.update(float(rng.binomial(1, drift_p)))
        if detector.drift_detected() and drift_pos is None:
            drift_pos = n_stable + i
    
    state = detector.get_state()
    detected = drift_pos is not None
    print(f"  {name:20s} | Drift={'YES' if detected else 'NO ':3s} | Position={drift_pos if drift_pos else 'N/A':>6} | Total drifts={state['n_drifts']}")
    return detected, drift_pos

print("=" * 70)
print("  DRIFT DETECTOR VERIFICATION")
print("  Stream: 500 stable samples (p=0.05) + 500 drift samples (p=0.60)")
print("=" * 70)

results = {}
detectors = [
    (ADWINDetector(delta=0.01), 'ADWIN'),
    (DDMDetector(), 'DDM'),
    (EDDMDetector(), 'EDDM'),
    (PageHinkleyDetector(), 'Page-Hinkley'),
    (UnsupervisedDriftDetector(delta=0.01), 'Unsupervised ADWIN'),
]

for det, name in detectors:
    detected, pos = test_detector(det, name)
    results[name] = (detected, pos)

print("=" * 70)

adwin_detected = results['ADWIN'][0]
assert adwin_detected, "FAIL: ADWIN should detect 5%->60% error shift!"
print(f"\nADWIN drift detection (primary detector): PASSED")

In [ ]:
det_cfg = {'detector': 'adwin', 'adwin': {'delta': 0.002, 'clock': 32, 'max_buckets': 5, 'min_window_length': 5, 'grace_period': 10}}
factory_det = create_detector(det_cfg)
assert factory_det.name == 'ADWIN', "FAIL: Factory should create ADWIN"
print(f"Detector factory: created {factory_det.name} with delta={factory_det.delta}")
print("Factory test: PASSED")

## 8. Synthetic Drift Injection

Tests all four drift types and verifies they modify data distributions as expected.

In [ ]:
gen = SyntheticDriftGenerator(seed=42)
X_base = np.random.RandomState(42).randn(2000, 5)
y_base = np.array(['ATTACK' if x > 0 else 'BENIGN' for x in X_base[:, 0]])

X_sudden, y_sudden = gen.sudden_drift(X_base, y_base, position=0.5, magnitude=3.0, label_flip_rate=0.15)
X_gradual, y_gradual = gen.gradual_drift(X_base, y_base, start=0.3, end=0.7, magnitude=3.0)
X_incr, y_incr = gen.incremental_drift(X_base, y_base, magnitude=2.0)
X_recur, y_recur = gen.recurring_drift(X_base, y_base, cycle_length=0.25, magnitude=3.0)

print("=" * 65)
print("  SYNTHETIC DRIFT INJECTION RESULTS")
print("=" * 65)

def check_drift(name, X_orig, X_mod, split_pos=1000):
    mean_before = X_mod[:split_pos, 0].mean()
    mean_after = X_mod[split_pos:, 0].mean()
    orig_mean = X_orig[:, 0].mean()
    shift = abs(mean_after - mean_before)
    print(f"  {name:15s}: mean_before={mean_before:+.3f}, mean_after={mean_after:+.3f}, shift={shift:.3f}")
    return shift

s1 = check_drift('Sudden', X_base, X_sudden)
s2 = check_drift('Gradual', X_base, X_gradual)
s3 = check_drift('Incremental', X_base, X_incr)
s4 = check_drift('Recurring', X_base, X_recur)

assert s1 > 0.3, f"FAIL: Sudden drift shift too small: {s1:.3f}"
np.testing.assert_array_equal(X_base, np.random.RandomState(42).randn(2000, 5), 
                               err_msg="FAIL: Original data was mutated!")

label_changes = np.sum(y_sudden != y_base)
print(f"\n  Sudden drift label flips: {label_changes}/{len(y_base)} ({label_changes/len(y_base)*100:.1f}%)")
assert label_changes > 0, "FAIL: No label flips in sudden drift!"

print(f"\n  All synthetic drift checks: PASSED")
print("=" * 65)

## 9. Streaming Simulation

In [ ]:
csv_stream = CSVStream(X_test_tmp, y_test_tmp, feature_names=pipeline_tmp.feature_columns)
batch_stream = BatchStream(X_test_tmp, y_test_tmp, batch_size=200, feature_names=pipeline_tmp.feature_columns)

print(f"CSVStream:   {len(csv_stream)} samples")
print(f"BatchStream: {len(batch_stream)} samples, batch_size=200")

sample_count = 0
for event in csv_stream:
    assert 'features' in event
    assert 'label' in event
    assert event['features'].shape[0] == X_test_tmp.shape[1]
    sample_count += 1
    if sample_count >= 10:
        break

batch_count = 0
total_in_batches = 0
for batch in batch_stream:
    assert 'features' in batch
    assert 'labels' in batch
    total_in_batches += len(batch['labels'])
    batch_count += 1

print(f"\nCSVStream iteration (first 10): PASSED")
print(f"BatchStream: {batch_count} batches, {total_in_batches} total samples")
assert total_in_batches == len(batch_stream), f"FAIL: Batch total {total_in_batches} != stream length {len(batch_stream)}"
print("Stream verification: PASSED")

## 10. Adaptation Strategies

Tests the three core strategies: static (no adaptation), periodic retraining, and drift-triggered retraining.

In [ ]:
print("=" * 65)
print("  ADAPTATION STRATEGY UNIT TESTS")
print("=" * 65)

static = StaticStrategy()
for i in range(1000):
    assert not static.should_retrain(i, drift_detected=(i % 100 == 0))
print(f"  StaticStrategy: never retrains after 1000 steps -> PASSED")

periodic = PeriodicStrategy(period=100)
periodic_retrain_positions = []
for i in range(550):
    if periodic.should_retrain(i, drift_detected=False):
        periodic_retrain_positions.append(i)
print(f"  PeriodicStrategy(100): retrained at positions {periodic_retrain_positions} -> {len(periodic_retrain_positions)} retrains")
assert len(periodic_retrain_positions) == 5, f"FAIL: Expected 5 periodic retrains, got {len(periodic_retrain_positions)}"
print(f"  PeriodicStrategy: PASSED")

drift_trig = DriftTriggeredStrategy(cooldown=50)
retrain_drift = []
for i in range(300):
    drift_signal = i in [100, 120, 160, 250]
    if drift_trig.should_retrain(i, drift_detected=drift_signal):
        retrain_drift.append(i)
print(f"  DriftTriggered(cooldown=50): retrained at {retrain_drift}")
assert 100 in retrain_drift, "FAIL: Should retrain at drift position 100"
assert 120 not in retrain_drift, "FAIL: Cooldown should prevent retrain at 120"
assert 160 in retrain_drift, "FAIL: Should retrain at 160 (past cooldown)"
print(f"  DriftTriggered: PASSED")

print("=" * 65)

## 11. End-to-End Adaptive Pipeline

The full research experiment: train an initial model, stream data through it, detect drift with ADWIN, and compare static vs periodic vs drift-triggered adaptation.

In [ ]:
from collections import defaultdict
import time

model_params = {'n_estimators': 50, 'max_depth': 6, 'verbose': -1, 'n_jobs': 1, 'num_threads': 1, 'class_weight': 'balanced'}

X_stream = np.vstack([X_train_tmp, X_test_tmp])
y_stream = np.concatenate([y_train_tmp, y_test_tmp])

train_size = min(1000, len(X_train_tmp))
X_init, y_init = X_stream[:train_size], y_stream[:train_size]
X_eval, y_eval = X_stream[train_size:], y_stream[train_size:]

strategies = {
    'static': StaticStrategy(),
    'periodic_500': PeriodicStrategy(period=500),
    'drift_triggered': DriftTriggeredStrategy(cooldown=200),
}

all_results = {}

for strat_name, strategy in strategies.items():
    detector = ADWINDetector(delta=0.002)
    manager = AdaptiveModelManager(
        algorithm='lightgbm',
        model_params=model_params,
        strategy=strategy,
        detector=detector,
        window_size=2000,
        random_seed=42,
    )
    manager.train_initial(X_init, y_init)
    
    predictions = []
    errors = []
    drift_positions = []
    retrain_positions = []
    
    t0 = time.perf_counter()
    for i in range(len(X_eval)):
        y_pred_i, is_correct, did_retrain = manager.process_sample(X_eval[i], y_eval[i], i)
        predictions.append(y_pred_i)
        errors.append(0.0 if is_correct else 1.0)
        if detector.drift_detected():
            drift_positions.append(i)
        if did_retrain:
            retrain_positions.append(i)
    elapsed = time.perf_counter() - t0
    
    predictions = np.array(predictions)
    final_metrics = compute_metrics(y_eval, predictions, positive_label='ATTACK')
    windowed_m = compute_windowed_metrics(y_eval, predictions, window_size=500, positive_label='ATTACK')
    
    all_results[strat_name] = {
        'metrics': final_metrics,
        'windowed': windowed_m,
        'errors': errors,
        'drift_positions': drift_positions,
        'retrain_positions': retrain_positions,
        'n_retrains': manager.n_retrains,
        'elapsed': elapsed,
    }

print("=" * 80)
print("  END-TO-END ADAPTIVE PIPELINE RESULTS")
print("=" * 80)
print(f"  {'Strategy':<20s} {'F1':>8s} {'Precision':>10s} {'Recall':>8s} {'MCC':>8s} {'Retrains':>10s} {'Time':>8s}")
print("-" * 80)
for name, res in all_results.items():
    m = res['metrics']
    print(f"  {name:<20s} {m['f1']:8.4f} {m['precision']:10.4f} {m['recall']:8.4f} {m['mcc']:8.4f} {res['n_retrains']:10d} {res['elapsed']:7.2f}s")
print("=" * 80)

n_drifts_detected = len(all_results['drift_triggered']['drift_positions'])
print(f"\nDrift events detected by ADWIN: {n_drifts_detected}")
print(f"Drift-triggered retrains:       {all_results['drift_triggered']['n_retrains']}")
print(f"Periodic retrains:              {all_results['periodic_500']['n_retrains']}")

## 12. Active Learning Verification

In [ ]:
print("=" * 60)
print("  ACTIVE LEARNING VERIFICATION")
print("=" * 60)

X_al_batch = X_test_rnd[:200]
y_al_batch = y_test_rnd[:200]

for StratClass, name in [(UncertaintySampling, 'Uncertainty'), (MarginSampling, 'Margin'), (RandomSampling, 'Random')]:
    strat = StratClass() if name != 'Random' else StratClass(seed=42)
    al_mgr = ActiveLearningManager(query_strategy=strat, label_budget_pct=0.1)
    
    X_sel, y_sel = al_mgr.query_and_label(lgbm_model, X_al_batch, y_al_batch)
    stats = al_mgr.get_stats()
    
    print(f"  {name:15s}: selected {len(X_sel):3d}/{len(X_al_batch)} samples ({stats['label_efficiency']:.1%} budget)")
    assert len(X_sel) == 20, f"FAIL: Expected 20 samples (10% of 200), got {len(X_sel)}"

pool_X, pool_y = al_mgr.get_labelled_pool()
assert len(pool_X) == 20, f"FAIL: Labelled pool size incorrect"
print(f"\n  Labelled pool size: {len(pool_X)}")
print("  Active Learning: PASSED")
print("=" * 60)

## 13. Visualizations

Generates publication-quality plots inline to prove the visualization pipeline works.

In [ ]:
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
sns.set_palette('colorblind')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Model Comparison
ax = axes[0, 0]
models = ['LightGBM\n(Random)', 'LightGBM\n(Temporal)', 'Random\nForest', 'LSTM']
f1_scores = [rnd_metrics['f1'], tmp_metrics['f1'], rf_metrics['f1'], lstm_metrics['f1']]
colors = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6']
bars = ax.bar(models, f1_scores, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Model Comparison (F1 Score)', fontsize=13, fontweight='bold')
for bar, score in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{score:.3f}', 
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='Target F1=0.9')
ax.legend(fontsize=9)

# Plot 2: Confusion Matrix (LightGBM Random)
ax = axes[0, 1]
cm = np.array(rnd_metrics['confusion_matrix'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=rnd_metrics['confusion_labels'], yticklabels=rnd_metrics['confusion_labels'])
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title('Confusion Matrix — LightGBM (Random Split)', fontsize=13, fontweight='bold')

# Plot 3: F1 Over Time (Adaptation Comparison)
ax = axes[1, 0]
strategy_colors = {'static': '#e74c3c', 'periodic_500': '#f39c12', 'drift_triggered': '#2ecc71'}
strategy_labels = {'static': 'Static (no adaptation)', 'periodic_500': 'Periodic (every 500)', 'drift_triggered': 'Drift-Triggered'}
for name, res in all_results.items():
    f1_over_time = [w['f1'] for w in res['windowed']]
    window_ids = [w['window_id'] for w in res['windowed']]
    ax.plot(window_ids, f1_over_time, 'o-', markersize=4, linewidth=1.5, 
            color=strategy_colors[name], label=strategy_labels[name])
    
for pos in all_results['drift_triggered']['drift_positions'][:5]:
    win_id = pos // 500
    if win_id < len(all_results['drift_triggered']['windowed']):
        ax.axvline(x=win_id, color='red', linestyle=':', alpha=0.3)

ax.set_xlabel('Window', fontsize=11)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title('Adaptation Strategy Comparison (F1 Over Time)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

# Plot 4: Error Rate with Drift
ax = axes[1, 1]
errors = all_results['drift_triggered']['errors']
window_size = 100
if len(errors) > window_size:
    rolling_error = pd.Series(errors).rolling(window=window_size, min_periods=1).mean()
else:
    rolling_error = pd.Series(errors)
ax.plot(range(len(rolling_error)), rolling_error, linewidth=1, alpha=0.8, color='#3498db', label=f'Rolling error (w={window_size})')
for dp in all_results['drift_triggered']['drift_positions'][:20]:
    ax.axvline(x=dp, color='red', linestyle='--', alpha=0.3)
for rp in all_results['drift_triggered']['retrain_positions']:
    ax.axvline(x=rp, color='green', linestyle='-', alpha=0.5)
ax.set_xlabel('Stream Position', fontsize=11)
ax.set_ylabel('Error Rate', fontsize=11)
ax.set_title('Error Rate & Drift Events (Drift-Triggered)', fontsize=13, fontweight='bold')
if all_results['drift_triggered']['drift_positions']:
    ax.axvline(x=-1, color='red', linestyle='--', alpha=0.5, label='Drift detected')
if all_results['drift_triggered']['retrain_positions']:
    ax.axvline(x=-1, color='green', linestyle='-', alpha=0.5, label='Retrain')
ax.legend(fontsize=9)

plt.suptitle('ADAPT-IDS: Pipeline Verification Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Visualization: PASSED")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Synthetic Drift Visualization
ax = axes[0, 0]
feature_idx = 0
ax.plot(X_base[:, feature_idx], alpha=0.3, label='Original', linewidth=0.5)
ax.plot(X_sudden[:, feature_idx], alpha=0.5, label='Sudden Drift', linewidth=0.5)
ax.axvline(x=1000, color='red', linestyle='--', label='Drift Point')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Feature Value')
ax.set_title('Sudden Drift Injection (Feature 0)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

# Plot 2: Gradual Drift
ax = axes[0, 1]
ax.plot(X_base[:, feature_idx], alpha=0.3, label='Original', linewidth=0.5)
ax.plot(X_gradual[:, feature_idx], alpha=0.5, label='Gradual Drift', linewidth=0.5, color='orange')
ax.axvline(x=600, color='red', linestyle='--', alpha=0.5, label='Drift Start')
ax.axvline(x=1400, color='red', linestyle=':', alpha=0.5, label='Drift End')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Feature Value')
ax.set_title('Gradual Drift Injection (Feature 0)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

# Plot 3: Class Distribution
ax = axes[1, 0]
label_counts = pd.Series(labels).value_counts()
label_counts.plot.barh(ax=ax, color=sns.color_palette('colorblind', len(label_counts)))
ax.set_xlabel('Count')
ax.set_title('Synthetic Dataset Class Distribution', fontsize=13, fontweight='bold')
for i, (label, count) in enumerate(label_counts.items()):
    ax.text(count + 20, i, f'{count} ({count/N_SAMPLES*100:.1f}%)', va='center', fontsize=10)

# Plot 4: Feature Importance (LightGBM)
ax = axes[1, 1]
importances = lgbm_model.feature_importances()
if importances is not None:
    feat_names = pipeline_rnd.feature_columns
    imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
    imp_df = imp_df.sort_values('importance', ascending=True).tail(10)
    ax.barh(imp_df['feature'], imp_df['importance'], color='#3498db')
    ax.set_xlabel('Importance')
    ax.set_title('Top 10 Feature Importances (LightGBM)', fontsize=13, fontweight='bold')

plt.suptitle('ADAPT-IDS: Data & Model Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Advanced visualizations: PASSED")

## 14. Detector Sensitivity Analysis

Sweeps ADWIN's delta parameter to show detection sensitivity vs false alarm tradeoff.

In [ ]:
deltas = [0.0001, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.05, 0.1, 0.5]
rng_sens = np.random.RandomState(42)

sensitivity_results = []
for delta in deltas:
    det = ADWINDetector(delta=delta)
    first_detection = None
    total_drifts = 0
    
    for i in range(300):
        det.update(float(rng_sens.binomial(1, 0.05)))
    
    for i in range(700):
        det.update(float(rng_sens.binomial(1, 0.5)))
        if det.drift_detected():
            total_drifts += 1
            if first_detection is None:
                first_detection = 300 + i
    
    rng_sens = np.random.RandomState(42)  # reset for next delta
    sensitivity_results.append({
        'delta': delta,
        'first_detection': first_detection,
        'total_drifts': total_drifts,
        'delay': (first_detection - 300) if first_detection else None,
    })

sens_df = pd.DataFrame(sensitivity_results)
print("=" * 55)
print("  ADWIN SENSITIVITY ANALYSIS")
print("=" * 55)
print(sens_df.to_string(index=False))
print("=" * 55)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

detected = sens_df[sens_df['delay'].notna()]
ax1.semilogx(detected['delta'], detected['delay'], 'o-', color='#e74c3c', markersize=8)
ax1.set_xlabel('Delta (log scale)', fontsize=11)
ax1.set_ylabel('Detection Delay (samples)', fontsize=11)
ax1.set_title('ADWIN: Detection Delay vs Delta', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.semilogx(sens_df['delta'], sens_df['total_drifts'], 's-', color='#3498db', markersize=8)
ax2.set_xlabel('Delta (log scale)', fontsize=11)
ax2.set_ylabel('Total Drift Alarms', fontsize=11)
ax2.set_title('ADWIN: Alarm Count vs Delta', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.suptitle('Drift Detector Sensitivity Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 15. Feature Analysis

In [ ]:
available_features = select_available_features(df)
print(f"Available CIC-IDS2017 features: {len(available_features)} / 68 recommended")
print(f"Features: {available_features}")

stats = compute_feature_stats(df, available_features)
print(f"\nFeature statistics (sample):")
print(stats[['mean', '50%', 'std', 'skewness', 'kurtosis']].head(8).to_string(float_format='{:.2f}'.format))

## 16. Complete Verification Summary

Final pass/fail report for every component.

In [ ]:
from collections import OrderedDict

checks = OrderedDict()

checks['Dependencies imported'] = True
checks['ADAPT-IDS modules imported'] = True

checks['Synthetic data generated'] = df.shape[0] == N_SAMPLES
checks['NaN/Inf injected'] = np.isinf(df.select_dtypes('number')).sum().sum() > 0

checks['Preprocessing: NaN removed'] = np.isnan(X).sum() == 0
checks['Preprocessing: Inf removed'] = np.isinf(X).sum() == 0
checks['Preprocessing: Binary labels'] = set(y) == {'ATTACK', 'BENIGN'}
checks['Preprocessing: Leakage-safe transform'] = X_t.shape[1] == X.shape[1]

checks['Temporal split: integrity'] = train_max_ts < test_min_ts

checks['LightGBM: trains and predicts'] = rnd_metrics['f1'] > 0
checks['LightGBM: F1 > 0.70 (random)'] = rnd_metrics['f1'] > 0.70
checks['Random Forest: trains and predicts'] = rf_metrics['f1'] > 0
checks['LSTM: trains and predicts'] = lstm_metrics['f1'] > 0

checks['Model save/load: LightGBM'] = True  # would have thrown assert earlier
checks['Model save/load: Random Forest'] = True
checks['Model save/load: LSTM'] = True

checks['Windowed metrics computed'] = len(windowed) > 0

checks['ADWIN: detects drift'] = results['ADWIN'][0]
checks['DDM: initializes'] = True
checks['EDDM: initializes'] = True
checks['Page-Hinkley: initializes'] = True
checks['Detector factory'] = factory_det.name == 'ADWIN'

checks['Synthetic drift: sudden'] = s1 > 0.3
checks['Synthetic drift: gradual'] = s2 > 0
checks['Synthetic drift: incremental'] = s3 > 0
checks['Synthetic drift: recurring'] = s4 > 0
checks['Synthetic drift: no mutation'] = True

checks['CSVStream: iterates'] = sample_count == 10
checks['BatchStream: all samples'] = total_in_batches == len(batch_stream)

checks['StaticStrategy: no retrains'] = True
checks['PeriodicStrategy: correct count'] = len(periodic_retrain_positions) == 5
checks['DriftTriggered: respects cooldown'] = True

checks['Adaptive pipeline: runs end-to-end'] = len(all_results) == 3
checks['Active learning: budget correct'] = True

n_pass = sum(checks.values())
n_total = len(checks)

print("\n" + "=" * 65)
print("  ADAPT-IDS VERIFICATION REPORT")
print("=" * 65)
for check_name, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    icon = '+' if passed else 'X'
    print(f"  [{icon}] {check_name:<45s} {status}")
print("=" * 65)
print(f"\n  TOTAL: {n_pass}/{n_total} checks passed")

if n_pass == n_total:
    print("\n  " + "*" * 45)
    print("  *  ALL CHECKS PASSED — PIPELINE VERIFIED  *")
    print("  " + "*" * 45)
else:
    failed = [name for name, passed in checks.items() if not passed]
    print(f"\n  FAILED CHECKS: {failed}")

print(f"\n  Platform: {platform.system()} {platform.machine()}")
print(f"  Python:   {sys.version.split()[0]}")
print(f"  Torch:    {torch.__version__}")
print(f"  LightGBM: {lgb.__version__}")

---

## Appendix: Quick Reference

### Key Metrics Explained (for someone new to cybersecurity)

| Metric | What It Means | Why It Matters for IDS |
|--------|--------------|----------------------|
| **F1 Score** | Harmonic mean of precision and recall | Balances catching attacks vs false alarms |
| **Precision** | Of all "attack" predictions, how many were real attacks | Low precision = too many false alarms |
| **Recall** | Of all real attacks, how many were detected | Low recall = missed attacks (dangerous!) |
| **FPR** (False Positive Rate) | % of benign traffic flagged as attacks | High FPR = alert fatigue for analysts |
| **FNR** (False Negative Rate) | % of attacks that slip through | High FNR = security breaches |
| **MCC** | Matthews Correlation Coefficient (-1 to +1) | Works well even with imbalanced classes |
| **Concept Drift** | Attack patterns change over time | Models trained on old data miss new attacks |
| **ADWIN** | Adaptive Windowing algorithm | Detects when error rate distribution changes |

### What the Results Mean

- **Random split F1 >> Temporal split F1**: This proves drift matters! A model tested on shuffled data looks great, but fails on chronologically newer data.
- **Drift-triggered retraining**: Only retrains when ADWIN detects the model's error pattern has changed. This is efficient — fewer retrains than periodic, but targeted when they matter.
- **Active learning**: Instead of labeling every sample, we only label the most informative ones (10% budget), saving analyst time.